# Lab 0 — Does the number hold up?

**MA553 · Session 1 · 9 September 2026**

Name:

---

`result_card.md` in this folder reports that gradient boosting beats logistic
regression by 3.5 percentage points on a medical classification task.

In this lab you find out how much that number is worth. Work through the
steps in order. Do not skip ahead to the later cells — some of the value
comes from committing to a prediction before you see the answer.

## Step 0 — Setup

Run this cell. If it fails, run `python check_setup.py` from the top of the
repository, or switch to the Colab version.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
print(f"{X.shape[0]} patients, {X.shape[1]} measurements each")

## Step 1 — Reproduce the reported result

The code below is what the result card released. It uses one particular
split of the data, fixed by `random_state=10`.

Run it, and check the difference against the 3.5 percentage points on the
card.

In [ ]:
SEED = 10

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.01, max_iter=200),
).fit(X_train, y_train)

proposed = GradientBoostingClassifier(random_state=SEED).fit(X_train, y_train)

baseline_accuracy = baseline.score(X_test, y_test) * 100
proposed_accuracy = proposed.score(X_test, y_test) * 100

print(f"Test set size:     {len(y_test)} patients")
print(f"Baseline:          {baseline_accuracy:.1f}%")
print(f"Proposed:          {proposed_accuracy:.1f}%")
print(f"Difference:        {proposed_accuracy - baseline_accuracy:+.1f} percentage points")

**Answer here.** Did you reproduce the reported number? If not, what did you
get, and by how much does it differ?

*Your answer:*

## Step 2 — Predict, before you run anything

`random_state=10` decided which patients ended up in the training set and
which in the test set. Nothing else about the methods would change if we
picked a different value.

Suppose we repeat the whole thing with 20 different splits.

**Write your predictions in the cell below, and do not change them
afterwards.** You are marked on the reasoning, never on the accuracy.

**My predictions.**

1. The difference will range roughly from ____ to ____ percentage points.
2. Out of 20 splits, gradient boosting will win about ____ times.

**My reasoning:**

*Your answer:*

## Step 3 — Run it across many splits

Now do exactly what the result card did, 20 times over, changing only the
split.

In [ ]:
def compare(seed, tuned_baseline=False):
    """Train both methods on one split and return the difference in
    test accuracy, in percentage points (positive = proposed method wins)."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    if tuned_baseline:
        # The baseline now gets a search over its main setting,
        # chosen using the training data only.
        baseline = GridSearchCV(
            make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
            {"logisticregression__C": [0.01, 0.1, 1, 10, 100]},
            cv=5,
        ).fit(X_train, y_train)
    else:
        # The baseline as the result card ran it: one fixed setting.
        baseline = make_pipeline(
            StandardScaler(), LogisticRegression(C=0.01, max_iter=200)
        ).fit(X_train, y_train)

    proposed = GradientBoostingClassifier(random_state=seed).fit(X_train, y_train)
    return (proposed.score(X_test, y_test) - baseline.score(X_test, y_test)) * 100


seeds = range(20)
differences = np.array([compare(s) for s in seeds])

print("Difference per split (percentage points):")
print(np.round(differences, 1))
print()
print(f"Mean:                {differences.mean():+.1f}")
print(f"Standard deviation:  {differences.std():.1f}")
print(f"Range:               {differences.min():+.1f} to {differences.max():+.1f}")
print(f"Proposed method won: {(differences > 0).sum()} of {len(differences)} splits")

In [ ]:
plt.figure(figsize=(8, 3.2))
plt.axhline(0, color="grey", linewidth=1)
plt.plot(list(seeds), differences, "o", color="#8C1D2C")
plt.axhline(differences.mean(), color="#8C1D2C", linestyle="--", linewidth=1,
            label=f"mean {differences.mean():+.1f}")
plt.xlabel("split (random_state)")
plt.ylabel("difference (percentage points)")
plt.title("Same methods, same data, different splits")
plt.legend()
plt.tight_layout()
plt.show()

**Answer here.**

1. How does the reported 3.5 compare with the other 19 splits?
2. How far off was your prediction, and why do you think that is?

*Your answer:*

## Step 4 — Give the baseline the same effort

Look again at the code in Step 1. The proposed method is gradient boosting
with its defaults. The baseline is logistic regression with `C=0.01` — one
particular setting, chosen by someone, and never varied.

`compare(seed, tuned_baseline=True)` gives the baseline a small search over
that setting, using only the training data. Nothing else changes.

In [ ]:
differences_tuned = np.array([compare(s, tuned_baseline=True) for s in seeds])

print("Difference per split, with the baseline tuned (percentage points):")
print(np.round(differences_tuned, 1))
print()
print(f"Mean:                {differences_tuned.mean():+.1f}")
print(f"Standard deviation:  {differences_tuned.std():.1f}")
print(f"Range:               {differences_tuned.min():+.1f} to {differences_tuned.max():+.1f}")
print(f"Proposed method won: {(differences_tuned > 0).sum()} of {len(differences_tuned)} splits")

In [ ]:
plt.figure(figsize=(8, 3.2))
plt.axhline(0, color="grey", linewidth=1)
plt.plot(list(seeds), differences, "o", color="#8C1D2C", label="baseline as released")
plt.plot(list(seeds), differences_tuned, "s", color="#26547C", label="baseline tuned")
plt.xlabel("split (random_state)")
plt.ylabel("difference (percentage points)")
plt.title("What happens when both sides get the same effort")
plt.legend()
plt.tight_layout()
plt.show()

**Answer here.** What happened to the advantage, and why? Which of the four
questions from today does this failure belong to?

*Your answer:*

## Step 5 — Write the claim

Go back to the sentence on the result card:

> Gradient boosting improves accuracy by 3.5 percentage points over logistic
> regression on the Wisconsin breast-cancer data.

**1. Write a claim your own results support.** Say what was compared, on
what data, over how many splits, under what conditions, and how much the
result varied.

*Your claim:*

**2. Name one thing your claim still does not establish.** Use the four
questions — measure, compare, scope, quantify — to find it.

*Your limitation:*

## Before you submit

- Restart the kernel and run every cell from top to bottom.
- Check that all five **Answer here** cells are filled in.
- If you used an AI assistant, add a short note saying what it produced and
  what you changed.
- Submit through Moodle before the next session.